In [8]:
import os

print("Contents of /kaggle/input/datasets/doppeltilde/hotdog-nothotdog/hotdog-nothotdog:")
print(os.listdir('/kaggle/input/datasets/doppeltilde/hotdog-nothotdog/hotdog-nothotdog'))


Contents of /kaggle/input/datasets/doppeltilde/hotdog-nothotdog/hotdog-nothotdog:
['test', 'train']


In [9]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from tqdm import tqdm
import torch.multiprocessing as mp
import datetime
from PIL import Image, UnidentifiedImageError


def is_valid_image_file(filepath: str) -> bool:
    try:
        with Image.open(filepath) as img:
            img.load()
            print(f"Valid Image: {filepath}")
        return True
    except (UnidentifiedImageError, OSError, IOError):
        print(f"Skipping invalid/corrupted image: {filepath}")
        return False
    except Exception as e:
        print(f"Unexpected error with file {filepath}: {type(e).__name__}")
        return False


def main():
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print(f"Using device: {device} - {torch.cuda.get_device_name(0)}")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        device = torch.device("mps")
        print(f"Using device: {device}")
    else:
        device = torch.device("cpu")
        print(f"Using device: {device}")

    transform = transforms.Compose(
        [
            transforms.Resize(456),
            transforms.CenterCrop(380),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
        ]
    )

    train_dataset = datasets.ImageFolder(
        root="/kaggle/input/datasets/doppeltilde/hotdog-nothotdog/hotdog-nothotdog/train",
        transform=transform,
        is_valid_file=is_valid_image_file,
    )

    print(
        f"Dataset successfully loaded: {len(train_dataset)} valid images, "
        f"{len(train_dataset.classes)} classes."
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=16,
        shuffle=True,
        num_workers=6,
        pin_memory=True if device.type in ["cuda", "mps"] else False,
        persistent_workers=False,
    )

    model = timm.create_model(
        "tf_efficientnet_lite4", pretrained=True, num_classes=len(train_dataset.classes)
    )
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-5,
    )

    use_amp = device.type in ["cuda", "mps"]
    scaler = torch.amp.GradScaler(enabled=use_amp)

    os.makedirs("checkpoints", exist_ok=True)

    num_epochs = 5
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in tqdm(train_loader):
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast(device_type=device.type, enabled=use_amp):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_dataset)
        print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

        checkpoint_name = f"checkpoints/checkpoint_epoch_{epoch+1}.pth"
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "classes": train_dataset.classes,
            },
            checkpoint_name,
        )
        print(f"Saved checkpoint: {checkpoint_name}")

    print("Training completed.")

    model_filename = f"efficientnet_lite4_final_finetuned.pth"

    os.makedirs("tflite", exist_ok=True)
    checkpoint_path = f"tflite/{model_filename}"

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "class_names": train_dataset.classes,
            "num_classes": len(train_dataset.classes),
            "timestamp": timestamp,
        },
        checkpoint_path,
    )

    print(f"PyTorch model saved to: {checkpoint_path}")


if __name__ == "__main__":
    mp.set_start_method("spawn", force=True)

    main()


Using device: cuda - Tesla T4
Valid Image: /kaggle/input/datasets/doppeltilde/hotdog-nothotdog/hotdog-nothotdog/train/hotdog/1.jpg
Valid Image: /kaggle/input/datasets/doppeltilde/hotdog-nothotdog/hotdog-nothotdog/train/hotdog/10.jpg
Valid Image: /kaggle/input/datasets/doppeltilde/hotdog-nothotdog/hotdog-nothotdog/train/hotdog/100.jpg
Valid Image: /kaggle/input/datasets/doppeltilde/hotdog-nothotdog/hotdog-nothotdog/train/hotdog/1000.jpg
Valid Image: /kaggle/input/datasets/doppeltilde/hotdog-nothotdog/hotdog-nothotdog/train/hotdog/1001.jpg
Valid Image: /kaggle/input/datasets/doppeltilde/hotdog-nothotdog/hotdog-nothotdog/train/hotdog/1002.jpg
Valid Image: /kaggle/input/datasets/doppeltilde/hotdog-nothotdog/hotdog-nothotdog/train/hotdog/1003.jpg
Valid Image: /kaggle/input/datasets/doppeltilde/hotdog-nothotdog/hotdog-nothotdog/train/hotdog/1004.jpg
Valid Image: /kaggle/input/datasets/doppeltilde/hotdog-nothotdog/hotdog-nothotdog/train/hotdog/1005.jpg
Valid Image: /kaggle/input/datasets/dopp

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


model.safetensors:   0%|          | 0.00/52.5M [00:00<?, ?B/s]

  0%|          | 0/188 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 6 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
100%|██████████| 188/188 [01:24<00:00,  2.23it/s]


Epoch [1/5], Loss: 0.6936
Saved checkpoint: checkpoints/checkpoint_epoch_1.pth


...

Epoch [3/5], Loss: 0.0646
Saved checkpoint: checkpoints/checkpoint_epoch_3.pth


100%|██████████| 188/188 [01:06<00:00,  2.85it/s]


Epoch [4/5], Loss: 0.0656
Saved checkpoint: checkpoints/checkpoint_epoch_4.pth


100%|██████████| 188/188 [01:05<00:00,  2.86it/s]


Epoch [5/5], Loss: 0.0338
Saved checkpoint: checkpoints/checkpoint_epoch_5.pth
Training completed.
PyTorch model saved to: tflite/efficientnet_lite4_final_finetuned.pth


In [3]:
import os
import torch
import timm
import litert_torch
from torchvision import transforms
from torchvision.datasets import ImageFolder
import traceback
from pathlib import Path

script_dir = Path(__file__).parent.resolve()


class NormalizedModel(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        x = x.float() / 255.0
        x = (x - torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 1, 3)) / torch.tensor(
            [0.229, 0.224, 0.225]
        ).view(1, 1, 1, 3)
        x = x.permute(0, 3, 1, 2)
        return self.model(x)


class LogitNormalizer(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.softmax = torch.nn.Softmax(dim=1)

    def forward(self, x):
        logits = self.model(x)
        return self.softmax(logits)


def main():
    print("Loading fine-tuned model for conversion...")

    transform = transforms.Compose(
        [
            transforms.Resize(456),
            transforms.CenterCrop(380),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
    )

    dataset = ImageFolder(root="dataset", transform=transform)
    class_names = dataset.classes

    checkpoint_path = script_dir / "tflite/efficientnet_lite4_final_finetuned.pth"
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)

    class_names = checkpoint["class_names"]
    num_classes = len(class_names)
    print(f"Loading model with {num_classes} classes from checkpoint.")

    base_model = timm.create_model(
        "tf_efficientnet_lite4",
        pretrained=False,
        num_classes=num_classes,
    )

    base_model.load_state_dict(checkpoint["model_state_dict"])

    print("Finetuned model loaded successfully.")

    labels_path = Path(script_dir / "tflite/labels.txt")
    labels_path.parent.mkdir(parents=True, exist_ok=True)

    with open(labels_path, "w", encoding="utf-8") as f:
        for name in class_names:
            f.write(name + "\n")

    print(f"labels.txt created successfully at: {labels_path}")
    print(f"Total labels written: {len(class_names)}")

    model = LogitNormalizer(NormalizedModel(base_model))
    model.eval()

    print("\nConverting model to TensorFlow Lite format...")

    try:
        sample_input = torch.randn(1, 380, 380, 3)

        edge_model = litert_torch.convert(model, (sample_input,))

        os.makedirs("tflite", exist_ok=True)
        tflite_path = script_dir / "tflite/model.tflite"

        edge_model.export(tflite_path)

        print(f"Conversion successful!")
        print(f"TensorFlow Lite model saved to: {tflite_path}")

    except Exception as e:
        print(f"\nLiteRT conversion failed: {e}")
        traceback.print_exc()
        return


if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'torchao.quantization.pt2e'